# Lecture 7: closing the model

**ECU44291 Quantitative Macroeconomics, Week 8.**

Until now the household took the wage, the interest rate and the pension as given. Today they come from the rest of the economy: firms set prices from the capital stock, and the capital stock is what households save. Everything here is on the two-period OLG economy from Lecture 2, where each step is instant and there is a closed form to check against. The sixty-cohort version is on the slides, and building it is M3.

This is the complete notebook, and I walk through it in the lecture. Before each code cell there is a question: answer it before you run the cell.

In [ ]:
import numpy as np
from scipy.optimize import brentq

## The two-period OLG economy, as in Lecture 2

Each period one young and one old household are alive; a period is a generation, about 30 years. The young work (so $L = 1$), pay a labour tax $\tau_L = 0.2$ and a pension contribution $\tau_p$, and save $s$. The old consume their savings at the gross return $R$, plus the pension $b$. Log utility, $\beta = 0.9$; the firm is Cobb-Douglas with $\alpha = 0.3$, $\delta = 0$.

- Household: $\max \ln c_y + \beta \ln c_o$ subject to $c_y + s = (1-\tau_L-\tau_p)\,w$ and $c_o = R\,s + b$. The Euler equation $1/c_y = \beta R / c_o$ and the two budgets give
$$s = \frac{\beta}{1+\beta}\,(1 - \tau_L - \tau_p)\, w \;-\; \frac{b}{(1+\beta)\, R}.$$
- Firm: $r = \alpha K^{\alpha-1} - \delta$, $w = (1-\alpha) K^{\alpha}$; the return to saving after the capital tax $\tau_K = 0.15$ is $R = 1 + (1-\tau_K)\, r$.
- Pension: $b = \tau_p\, w$ (one worker pays, one retiree receives). Capital market: $K = s$.

The functions below are the Lecture 2 ones, unchanged (they also allow transfers $t_y$, $t_o$, which are zero here). One difference in names: the Lecture 2 code calls the rental rate, the marginal product of capital, `q`; it is $r + \delta$, so `q - delta` in the code is $r$.

In [ ]:
def params(**overrides):
    p = dict(alpha=0.3, A=1.0, beta=0.9, delta=0.0, tau_L=0.2, tau_K=0.15,
             t_y=0.0, t_o=0.0, tau_p=0.0, N_y=1.0, N_o=1.0)
    p.update(overrides)
    return p

def prices(K, p):
    L = p["N_y"]
    q = p["alpha"] * p["A"] * K ** (p["alpha"] - 1) * L ** (1 - p["alpha"])
    w = (1 - p["alpha"]) * p["A"] * K ** p["alpha"] * L ** (-p["alpha"])
    R = 1 + (1 - p["tau_K"]) * (q - p["delta"])
    return q, w, R

def pension_benefit(w, p):
    return p["tau_p"] * w * p["N_y"] / p["N_o"]

def savings(w, R, p):
    income_y = (1 - p["tau_L"] - p["tau_p"]) * w + p["t_y"]
    income_o = p["t_o"] + pension_benefit(w, p)
    return (p["beta"] * R * income_y - income_o) / ((1 + p["beta"]) * R)

def capital_supply(K, p):
    _, w, R = prices(K, p)
    return p["N_y"] * savings(w, R, p)

def solve_by_root(p):
    return brentq(lambda K: capital_supply(K, p) - K, 1e-3, 2.0, xtol=1e-12)

**Question before running:** with no pension the young save the share $\frac{\beta}{1+\beta}(1-\tau_L) = \frac{0.9}{1.9} \times 0.8$ of the wage. What should `K / w` print?

In [ ]:
p0 = params()
K_base = solve_by_root(p0)
_, w0, R0 = prices(K_base, p0)
print(f"K = {K_base:.4f}   w = {w0:.4f}   R = {R0:.4f}   K / w = {K_base / w0:.4f}")

## Task 1: the pension, once prices move

On the slides, a pension of $\tau_p = 0.3$ cut saving to 0.0620 at the old prices, and then capital and prices moved together, 0.0620, 0.0548, 0.0537, ..., until $K$ stopped changing. Here `brentq` finds where that process ends, as in Lecture 2. Print both steady states side by side.

**Question before running:** the old now receive a pension. In the new steady state, is their consumption $c_o$ higher or lower than without it?

In [ ]:
# your code here


**Read it with the saving rule.** The gap between 0.0620 and 0.0535 is general equilibrium: less capital, a lower wage, so less is saved out of wages, and the pension (a share of the wage) is smaller. The higher $R$ raises saving only through the second term, a little. And $c_o = R\,s + b$ falls: the return is higher, but on a third of the savings.

## Task 2 (homework): a labour tax, at old and new prices

Raise the labour tax from $\tau_L = 0.2$ to $0.3$, with no pension. Compute saving at the old wage, then in the new equilibrium, and government revenue $G = \tau_L w + \tau_K\, r\, s$ in both steady states. Why does revenue rise by less than 50 percent?

In [ ]:
# your code here


## Task 3: the same equilibrium by Gauss-Seidel

`brentq` worked because the saving rule is a formula. With sixty cohorts it is not, and the general method is the loop on method card 7: guess $K$, compute prices, let households save, add up, and use what they supply as the next guess, $K \leftarrow K_{\text{new}}$. It is Lecture 1's fixed-point iteration. Here `capital_supply(K, p)` does the middle three steps in one call.

**Question before running:** the map $K \mapsto K_{\text{new}}$ has slope about 0.3 at the steady state, so each step removes about 70% of the error. From $K_0 = 0.5$, roughly how many steps to get within $10^{-10}$?

In [ ]:
# your code here


**Check the answer.** Households must supply exactly the capital stock the loop returns, and it must be the one `brentq` found.

In [ ]:
# your code here


## Task 4: damping

With sixty cohorts the plain loop cycles (see the slides). The fix is to move only part of the way to $K_{\text{new}}$: $K \leftarrow \lambda K_{\text{new}} + (1-\lambda) K$, with $\lambda \in (0, 1]$. It is one new argument and one changed line; $\lambda = 1$ is the loop above.

**Question before running:** damping turns the map's slope $m$ into $\lambda m + (1-\lambda)$. Here $m = 0.3$. At $\lambda = 0.5$, more or fewer than 18 steps?

In [ ]:
# your code here


Here damping is not needed and only costs steps: the slope 0.3 becomes 0.65. With sixty cohorts the slope is about $-1.3$, and damping is what makes the loop converge at all.

## Task 5: calibrating $\beta$ by bracketing root finding

$K/Y$ rises with $\beta$. To find the $\beta$ that hits a target, solve $K/Y(\beta) - \text{target} = 0$: root finding, where every evaluation solves the whole economy. Here each one is instant; with sixty cohorts each is a full Gauss-Seidel loop, which is why the method must be safe and frugal.

**Question before running:** in this economy $K/Y = 0.2653$ at $\beta = 0.9$. For a target of $0.25$, is the calibrated $\beta$ above or below $0.9$?

In [ ]:
# your code here


## The sixty-cohort economy: M3

The slides show what happens with sixty cohorts of your household: the map slopes down, so the plain loop cycles and needs damping; a grid choice makes the map a staircase, so build on your continuous-choice household from Part C; and $\beta$ is calibrated exactly as in Task 5, with each evaluation a full equilibrium. Building it is M3, due Tuesday 24 November at 17:00, so there is no code for it here.

## Before the next lecture

- **M2, the proposal: Friday 13 November, 17:00.** Own-idea conversations by Thursday 12 November.
- Task 2 above, the labour tax.
- Method card 7.
- Next lecture, Thursday 12 November: calibration on Irish data.